In [1]:
# ============================================================
# RailGuard - YOLO Model Comparison Table
# Input: 3 Ultralytics results.csv files
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# CHANGE ONLY THESE PATHS
# ------------------------------------------------------------
CSV_FILES = [
    "/content/8results.csv",       # YOLOv8
    "/content/11results.csv",    # YOLO11
    "/content/26results.csv"     # YOLO26
]

MODEL_NAMES = [
    "YOLOv8",
    "YOLO11",
    "YOLO26"
]


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def find_col(df, possible_names):
    for name in possible_names:
        if name in df.columns:
            return name
    return None


def get_best_result(csv_path):

    df = pd.read_csv(csv_path)

    # Remove whitespace from column names
    df.columns = [c.strip() for c in df.columns]

    precision_col = find_col(df, [
        "metrics/precision(B)",
        "metrics/precision"
    ])

    recall_col = find_col(df, [
        "metrics/recall(B)",
        "metrics/recall"
    ])

    map50_col = find_col(df, [
        "metrics/mAP50(B)",
        "metrics/mAP50"
    ])

    map5095_col = find_col(df, [
        "metrics/mAP50-95(B)",
        "metrics/mAP50-95"
    ])

    epoch_col = find_col(df, [
        "epoch"
    ])

    required = [
        precision_col,
        recall_col,
        map50_col,
        map5095_col
    ]

    if any(x is None for x in required):
        print("\nColumns found in:", csv_path)
        print(df.columns.tolist())
        raise ValueError("Could not find required YOLO metric columns.")

    # --------------------------------------------------------
    # Select BEST epoch based on mAP50-95
    # --------------------------------------------------------
    best_idx = df[map5095_col].idxmax()
    best = df.loc[best_idx]

    precision = float(best[precision_col]) * 100
    recall = float(best[recall_col]) * 100
    map50 = float(best[map50_col]) * 100
    map5095 = float(best[map5095_col]) * 100

    # F1 score
    f1 = 2 * precision * recall / (precision + recall)

    epoch = int(best[epoch_col]) if epoch_col else best_idx + 1

    return {
        "Precision": precision,
        "Recall": recall,
        "mAP50": map50,
        "mAP5095": map5095,
        "F1": f1,
        "Epoch": epoch
    }


# ------------------------------------------------------------
# Read all three models
# ------------------------------------------------------------
results = []

for model_name, csv_file in zip(MODEL_NAMES, CSV_FILES):

    if not Path(csv_file).exists():
        raise FileNotFoundError(
            f"\nCSV not found:\n{csv_file}\n"
            "Check the path."
        )

    r = get_best_result(csv_file)

    results.append({
        "Model": model_name,
        **r
    })


# ------------------------------------------------------------
# Print terminal-style table
# ------------------------------------------------------------

WIDTH = 86

print("\n")
print("=" * WIDTH)
print(" " * 20 + "RAILGUARD - MODEL COMPARISON")
print(" " * 13 + "(Best Validation Results by mAP@50-95)")
print("=" * WIDTH)

print(
    f"| {'Metric':<22} | "
    f"{'YOLOv8':^17} | "
    f"{'YOLO11':^17} | "
    f"{'YOLO26':^17} |"
)

print("-" * WIDTH)


rows = [
    ("Precision (%)", "Precision"),
    ("Recall (%)", "Recall"),
    ("mAP@50 (%)", "mAP50"),
    ("mAP@50-95 (%)", "mAP5095"),
    ("F1-Score (%)", "F1"),
    ("Best Epoch", "Epoch")
]

for label, key in rows:

    values = []

    for r in results:
        if key == "Epoch":
            values.append(f"{int(r[key]):>8}")
        else:
            values.append(f"{r[key]:>8.3f}")

    print(
        f"| {label:<22} | "
        f"{values[0]:^17} | "
        f"{values[1]:^17} | "
        f"{values[2]:^17} |"
    )

print("=" * WIDTH)


# ------------------------------------------------------------
# Winner for each metric
# ------------------------------------------------------------

print("\n")
print("=" * WIDTH)
print(" " * 27 + "BEST MODEL")
print("=" * WIDTH)

metrics = [
    ("Precision", "Precision"),
    ("Recall", "Recall"),
    ("mAP@50", "mAP50"),
    ("mAP@50-95", "mAP5095"),
    ("F1-Score", "F1")
]

for label, key in metrics:

    winner = max(results, key=lambda x: x[key])

    print(
        f"{label:<15} : "
        f"{winner['Model']:<8} "
        f"({winner[key]:.3f}%)"
    )

print("=" * WIDTH)



                    RAILGUARD - MODEL COMPARISON
             (Best Validation Results by mAP@50-95)
| Metric                 |      YOLOv8       |      YOLO11       |      YOLO26       |
--------------------------------------------------------------------------------------
| Precision (%)          |       43.194      |       44.674      |       43.848      |
| Recall (%)             |       37.667      |       41.720      |       37.209      |
| mAP@50 (%)             |       35.446      |       38.961      |       35.514      |
| mAP@50-95 (%)          |       12.903      |       14.513      |       13.254      |
| F1-Score (%)           |       40.242      |       43.146      |       40.257      |
| Best Epoch             |           47      |           46      |           50      |


                           BEST MODEL
Precision       : YOLO11   (44.674%)
Recall          : YOLO11   (41.720%)
mAP@50          : YOLO11   (38.961%)
mAP@50-95       : YOLO11   (14.513%)
F1-Score     